# A first insider detector: Isolation Forest on wallet features
Stats C161/C261, guest lecture, Block 3. **Preview of Week 5**: the algorithm is explained properly there; here we run it end to end.

**The data.** `data/wallet_features.csv`: one row per wallet, for every wallet that bought shares in one of five event groups
(US strikes on Iran, US x Iran ceasefire, Machado Nobel Peace Prize 2025, Maduro out, OpenAI browser) between April 2025 and March 2026.
The six features are computed over the wallet's **whole history** in the lab's trade files (1,644 markets, 8.4M trade rows).
Every buyer counts, whether it placed the order (maker) or hit one (taker). Column `confirmed` is `True` for the wallets that were
later publicly identified as insiders (press reports, an SSRN paper, court filings): our few known positives.

**Wallet ids are pseudonymous.** `w_` followed by ten hex digits, derived from the real address with a secret salt: the same
wallet always gets the same id, and the id cannot be turned back into an address. This course works with pseudonymous wallets
only. Do not try to identify the people behind any wallet.

| feature | meaning |
|---|---|
| `position_anomaly_ratio` | biggest single buy divided by the wallet's average spend per market. "Is this bet out of character?" |
| `market_notional_ratio` | the wallet's largest share of one market's total volume. "Did it move the market?" |
| `avg_buy_price` | volume-weighted price paid for shares. Insiders buy conviction at 0.2-0.6, not lottery tickets at 0.02 |
| `concentration` | share of all spending that went into one market. All-in = 1 |
| `wallet_age_days` | first to last trade seen. Fresh wallets are cheap to make |
| `unique_markets` | how many markets the wallet touched |

These six are one reasonable set, not the right one. None of them uses the market's outcome: a feature that knows who won has nothing left to detect.

## 1. Load the wallet table

In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 44)

W = pd.read_csv("data/wallet_features.csv", index_col="wallet")
FEATS = ["position_anomaly_ratio", "market_notional_ratio", "avg_buy_price", "concentration", "wallet_age_days", "unique_markets"]
SHORT = {"position_anomaly_ratio": "pos_ratio", "market_notional_ratio": "mkt_share", "avg_buy_price": "price", "concentration": "conc",
         "wallet_age_days": "age_d", "unique_markets": "markets", "trade_count": "trades", "total_notional": "total_$"}   # short names for display only
print(f"{len(W):,} wallets, {int(W.confirmed.sum())} of them publicly confirmed insiders")
W[FEATS + ["trade_count", "total_notional", "confirmed"]].head(8).rename(columns=SHORT)

55,952 wallets, 19 of them publicly confirmed insiders


,pos_ratio,mkt_share,price,conc,age_d,markets,trades,total_$,confirmed
wallet,,,,,,,,,
w_b6640fc6a7,0.365252,0.000322,0.576023,1.000000,1.598889,1,6,241.805798,False
w_ce85723407,1.501651,0.000362,0.936964,0.405506,49.930856,4,6,33.230088,False
w_b92e826e83,1.099906,0.000021,0.340639,0.483274,17.895579,4,7,43.640110,False
w_ab04538147,2.331724,0.000559,0.204462,0.388621,19.121771,6,10,135.093199,False
w_09a7eb8ca6,1.008677,0.000114,0.678542,0.077591,4.868113,13,13,64.902260,False
w_d041e2067b,3.502354,0.000001,0.804635,0.700471,27.958044,5,5,23.369995,False
w_07fdda55e6,1.258649,0.000476,0.923803,0.530658,3.153634,3,8,2951.797788,False
w_aca11e3e0e,2.643086,0.005882,0.957105,0.241208,14.600370,11,24,504.735026,False


## 2. Stage 1: set the bots aside
Market-making bots trade thousands of times across hundreds of markets. They are the most *anomalous* wallets by any
measure, and if we leave them in, the forest spends its attention on them and insiders disappear. So: a wallet with
500+ markets or 2,000+ trades is a bot for our purposes, and is scored 0.

In [2]:
W["is_bot"] = (W.unique_markets >= 500) | (W.trade_count >= 2000)
H = W[~W.is_bot].copy()
print(f"bots set aside: {W.is_bot.sum()}   humans to score: {len(H):,}")

bots set aside: 284   humans to score: 55,668


## 3. Stage 2: fit the Isolation Forest
200 random trees. Each tree isolates points with random axis-parallel cuts; a point that is alone after few cuts is anomalous.
`contamination=0.05` tells the model to call the top 5% "outliers". We ignore that binary call and keep the **ranking**.

In [3]:
from sklearn.ensemble import IsolationForest
X = H[FEATS].to_numpy(float)
for c in range(X.shape[1]):                          # fill the few missing values with the column median
    bad = ~np.isfinite(X[:, c]); X[bad, c] = np.nanmedian(X[~bad, c])

forest = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1).fit(X)
H["raw"] = forest.decision_function(X)               # negative = anomalous
H["score"] = 100 / (1 + np.exp(10 * H.raw))          # 0..100, 100 = most anomalous (the lab's display scale)
H["rank"] = H.raw.rank(method="first").astype(int)   # 1 = most anomalous
H = H.sort_values("rank")
print(f"fitted on {len(H):,} wallets")

ModuleNotFoundError: No module named 'sklearn'

## 4. The top 1%: who does the forest point at?

In [4]:
n = len(H); top1 = int(np.ceil(0.01 * n))
H["insider"] = np.where(H.confirmed, "*** confirmed", "")
print(f"top 1% = {top1} wallets. Confirmed insiders among them: {int(H.iloc[:top1].confirmed.sum())}")
H.iloc[:20][["score", "insider"] + FEATS + ["total_notional"]].round(3).rename(columns=SHORT)

top 1% = 557 wallets. Confirmed insiders among them: 0


KeyError: "['score'] not in index"

## 5. Where do the confirmed insiders land?
This is the only evaluation we can do with a handful of positive labels: **ranks**, not accuracy.

In [ ]:
lab = H[H.confirmed]
lab[["rank", "score"] + FEATS + ["total_notional"]].assign(percentile=(lab["rank"] / n * 100).round(2)).round(3).rename(columns=SHORT)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.6))
ax.hist(H["rank"], bins=100, color="#d2d2ef")
for _, r in lab.iterrows():
    ax.axvline(r["rank"], color="#c40000", lw=1.3)
ax.axvspan(0, top1, color="#3333b2", alpha=0.2)
ax.set_xlabel(f"anomaly rank among {n:,} wallets (1 = most anomalous); red = a confirmed insider; blue band = top 1%")
ax.set_ylabel("wallets per bin"); plt.tight_layout(); plt.show()

## 6. Precision at k, against chance and against a simple baseline
If we flag the top k% and hand the list to a human, how many confirmed insiders are on it? Compare with chance, and with the
simplest possible ranking: **biggest spenders first**. Always run a simple baseline; if your model does not beat it, say so.

In [ ]:
H["spend_rank"] = H.total_notional.rank(ascending=False, method="first")     # the simple baseline: rank by total spend
rows = []
for k in (0.5, 1, 2.5, 5, 10):
    cut = int(np.ceil(k / 100 * n))
    rows.append(dict(top_percent=k, wallets_flagged=cut, of=len(lab),
                     forest_catches=int((lab["rank"] <= cut).sum()),
                     biggest_spenders_catch=int((H.loc[lab.index, "spend_rank"] <= cut).sum()),
                     chance=round(len(lab) * k / 100, 2)))
pd.DataFrame(rows)

The biggest-spenders ranking wins, and that is the lesson, not a failure: the *confirmed* insiders are the ones the press
caught, and the press caught the biggest winners. Size finds them. The forest ranks by how **unusual** a wallet is, which is a
different question, and the insiders that neither ranking finds are the small, fresh ones (section 9).

## 7. One insider vs one ordinary trader
The highest-ranked confirmed insider next to a wallet from the middle of the ranking with a normal footprint.

In [ ]:
best = lab.iloc[0]
mid = H.iloc[n // 2 - 50: n // 2 + 50]
ordinary = mid[(mid.trade_count >= 10) & (mid.unique_markets >= 2)].iloc[0]
pd.DataFrame({f"confirmed insider ({best.name})": best, f"ordinary trader ({ordinary.name})": ordinary}).loc[["rank", "score"] + FEATS + ["trade_count", "total_notional"]].round(3)

## 8. The whole population in two dimensions (PCA, from Week 1)
Standardise the log of the six features, project to two components. Blue = flagged by the forest, red stars = confirmed insiders.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
Xs = H[FEATS].to_numpy(float)                        # H is sorted by rank now, so rebuild the matrix in that order
for c in range(Xs.shape[1]):
    bad = ~np.isfinite(Xs[:, c]); Xs[bad, c] = np.nanmedian(Xs[~bad, c])
Z = StandardScaler().fit_transform(np.log1p(np.clip(Xs, 0, None)))
P = PCA(2, random_state=0).fit(Z); pc = P.transform(Z)
flag = (H["rank"] <= top1).to_numpy(); li = H.confirmed.to_numpy()
fig, ax = plt.subplots(figsize=(9, 6.5))
ax.scatter(pc[~flag, 0], pc[~flag, 1], s=5, color="#9a9ad8", alpha=0.3, label="everyone else")
ax.scatter(pc[flag, 0], pc[flag, 1], s=22, color="#3333b2", label=f"top 1% ({top1})")
ax.scatter(pc[li, 0], pc[li, 1], s=160, marker="*", color="#c40000", edgecolor="k", lw=0.5, label=f"confirmed insiders ({li.sum()})")
ax.set_xlabel(f"PC1 ({P.explained_variance_ratio_[0]:.0%})"); ax.set_ylabel(f"PC2 ({P.explained_variance_ratio_[1]:.0%})"); ax.legend()
plt.tight_layout(); plt.show()

## 9. What the misses have in common
The confirmed insiders the forest did *not* flag. Look at `unique_markets`, `wallet_age_days` and `trade_count`.

In [ ]:
missed = lab[lab["rank"] > int(np.ceil(0.05 * n))]
missed[["rank", "unique_markets", "wallet_age_days", "trade_count", "concentration", "total_notional"]].round(2).rename(columns=SHORT)

A brand-new wallet that makes a handful of bets in one market and never trades again is **not anomalous**: thousands of
one-shot gamblers look exactly like that. Per-wallet features cannot separate a burner insider from a burner gambler.
What could is context that one wallet's rows do not carry: when the bets happened relative to the news, and what moved in the
related markets of the same event. That is where the lecture ends, and where your project could begin.

## 10. Save your ranking
One line. The file lands next to the wallet table; open it in Excel or hand it to a teammate.

In [ ]:
H.drop(columns=["insider", "spend_rank"]).to_csv("data/wallet_scores.csv")
print("saved data/wallet_scores.csv with", len(H), "rows")